In [3]:
!pip install faiss-cpu sentence-transformers

We will chunnk the data, both the cookbooks, flavor bible, and odorant to food data. We store the cookbooks separatley from the other sources, because they will be helpful in constructing recipes. The Flavor bible and odorant to food docs that we generated earlier will be useful in creating pairings.

In [16]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')


DATA_DIR = "/content/drive/MyDrive/Culinary Agent/rag_docs"
COOKBOOK_DIR = os.path.join(DATA_DIR, "Cook-book-collection")

INGREDIENT_DOCS = os.path.join(DATA_DIR, "ingredient_flavor_docs.jsonl")
ODORANT_DOCS = os.path.join(DATA_DIR, "odorant_docs.jsonl")
FLAVOR_BIBLE = os.path.join(DATA_DIR, "flavor_bible.txt.rtf")

# files we will generate
RECIPE_CHUNK_CACHE = os.path.join(DATA_DIR, "recipe_chunked_docs.jsonl")
PAIRING_CHUNK_CACHE = os.path.join(DATA_DIR, "pairing_chunked_docs.jsonl")
EMBED_CACHE = os.path.join(DATA_DIR, "embeddings.npy")
INDEX_PATH = os.path.join(DATA_DIR, "faiss.index")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
def chunk_text(text, chunk_size=220, overlap=40):

    words = text.split()
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = words[i:i+chunk_size]
        chunks.append(" ".join(chunk))

    return chunks

In [18]:
recipe_docs = []
pairing_docs = []

if (not os.path.exists(RECIPE_CHUNK_CACHE)) or (not os.path.exists(PAIRING_CHUNK_CACHE)):

    print("Building chunked corpus...")

    # Load JSONL docs
    for path in [INGREDIENT_DOCS, ODORANT_DOCS]:

        with open(path) as f:
            for line in f:
                pairing_docs.append(json.loads(line))

    # Flavor Bible
    with open(FLAVOR_BIBLE) as f:
        text = f.read()

    for chunk in chunk_text(text):

        pairing_docs.append({
            "type": "cookbook",
            "source": "flavor_bible",
            "text": chunk
        })

    # Cookbook collection
    for file in os.listdir(COOKBOOK_DIR):

        if not file.endswith(".txt"):
            continue

        path = os.path.join(COOKBOOK_DIR, file)

        with open(path) as f:
            text = f.read()

        for chunk in chunk_text(text):

            recipe_docs.append({
                "type": "cookbook",
                "source": file,
                "text": chunk
            })

    print("Total pairing docs:", len(pairing_docs))
    print("Total recipe docs:", len(recipe_docs))

    # add pairing docs to pairing cache
    with open(PAIRING_CHUNK_CACHE, "w") as f:
        for doc in pairing_docs:
            f.write(json.dumps(doc) + "\n")

    # add recipe docs to recipe cahce
    with open(RECIPE_CHUNK_CACHE, "w") as f:
        for doc in recipe_docs:
            f.write(json.dumps(doc) + "\n")

else:
  print("hi")

Building chunked corpus...
Total pairing docs: 2342
Total recipe docs: 4878


In [19]:

print("Computing recipe embeddings...")

model = SentenceTransformer("BAAI/bge-large-en-v1.5")

recipe_text = [d["text"] for d in recipe_docs]

recipe_vectors = model.encode(
    recipe_text,
    normalize_embeddings=True,
    show_progress_bar=True
)

recipe_vectors = np.array(recipe_vectors).astype("float32")

np.save(EMBED_CACHE, recipe_vectors)


Computing recipe embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

In [20]:
print("Computing pairing embeddings...")

pairings_text = [d["text"] for d in pairing_docs]

pairing_vectors = model.encode(
    pairings_text,
    normalize_embeddings=True,
    show_progress_bar=True
)

pairing_vectors = np.array(pairing_vectors).astype("float32")

np.save(EMBED_CACHE, pairing_vectors)

Computing pairing embeddings...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

In [21]:
print("Building Recipe FAISS index")

RECIPE_INDEX_PATH = os.path.join(DATA_DIR, "recipe_faiss.index")
PAIRING_INDEX_PATH = os.path.join(DATA_DIR, "pairing_faiss.index")

recipe_dim = recipe_vectors.shape[1]

recipe_index = faiss.IndexFlatIP(recipe_dim)

recipe_index.add(recipe_vectors)

faiss.write_index(recipe_index, RECIPE_INDEX_PATH)



Building Recipe FAISS index


In [22]:
print("Building Pairing FAISS index")

pairing_dim = pairing_vectors.shape[1]

pairing_index = faiss.IndexFlatIP(pairing_dim)

pairing_index.add(pairing_vectors)

faiss.write_index(pairing_index, PAIRING_INDEX_PATH)

Building Pairing FAISS index


In [23]:
RECIPE_META_PATH = os.path.join(DATA_DIR, "recipe_metadata.json")
PAIRING_META_PATH = os.path.join(DATA_DIR, "pairing_metadata.json")

print("Saving recipe metadata")

with open(RECIPE_META_PATH, "w") as f:
    json.dump(recipe_docs, f)

print("Recipe metadata saved")

Saving recipe metadata
Recipe metadata saved


In [24]:
print("Saving pairing metadata")

with open(PAIRING_META_PATH, "w") as f:
    json.dump(pairing_docs, f)

print("Pairing metadata saved")

Saving pairing metadata
Pairing metadata saved


In [27]:
## this code is for agent
pairing_index = faiss.read_index(os.path.join(DATA_DIR, "pairing_faiss.index"))
pairing_metadata = json.load(open(os.path.join(DATA_DIR, "pairing_metadata.json")))
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

def search(query, k=6):

    qvec = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = pairing_index.search(qvec, k)

    return [pairing_metadata[i] for i in ids[0]]

#example usage
results = search("why does basil pair well with tomato? Which odorants do they share?")

for r in results:
    print("\n---")
    print(r["text"][:400])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



---
Ingredient: basil

Key odorants: coumarin, methyl benzoate, 3-hexanone, geraniol, 2-ethyl-4-hydroxy-5-methyl-3(2h)-furanone, terpinolene

Flavor descriptors: chicken, characteristic odor similar to acetaldehyde, civet, citral, acquires terebinthinate odor upon oxidation, smoky, powerful grassy-green odor, strong odor, cherry, pine

These aroma compounds define the characteristic flavor of basil.

---
Odorant: 2-isobutylthiazole

Aroma descriptors:
metallic, tomato leaf, leaf, green, earthy, vegetable.

This odorant appears in tomato foods such as tomato.

Because it is shared across many leafy ingredients,
it contributes to flavor compatibility between them.


---
Ingredient: tomato

Key odorants: hexyl hexanoate, coumarin, 2-methylbutyraldehyde, (2e,4e)-deca-2,4-dienal, 3-methylthiopropanol, methyl benzoate

Flavor descriptors: chicken, characteristic odor similar to acetaldehyde, civet, citral, smoky, privet, powerful grassy-green odor, strong odor, cherry, pine

These aroma com

In [26]:
## this code is for agent
recipe_index = faiss.read_index(os.path.join(DATA_DIR, "recipe_faiss.index"))
recipe_metadata = json.load(open(os.path.join(DATA_DIR, "recipe_metadata.json")))
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

def search(query, k=6):

    qvec = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = recipe_index.search(qvec, k)

    return [recipe_metadata[i] for i in ids[0]]

#example usage
results = search("How to make soup with basil and tomato")

for r in results:
    print("\n---")
    print(r["text"][:400])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



---
saucepan. Simmer 30 minutes. Puree, along with the basil leaves, in small batches, in a blender or food processor. Return to saucepan and add cream and butter, while stirring over low heat. Garnish with extra basil leaves and serve with your favorite bread. Tomato Basil Soup 50 Veggie Latkes 2 medium russet potatoes 1 small carrot 1 small zucchini 3/4 cup shredded cabbage 1 small onion 1 large rib

---
unsalted butter salt and pepper to taste Combine tomatoes and stock in saucepan. Simmer 30 minutes, along with the basil leaves, in small batches, in a blender or food processor. Return to saucepan and add cream and butter, while stirring over low heat. Garnish with extra basil leaves and serve with your favorite bread. Tomato Basil Soup Veggie Latkes 2 medium russet potatoes 1 small carrot 1 smal

---
Swiss and then Parmesan cheese. Place bowls on cookie sheet and broil until cheese bubbles and browns slightly. French Onion Soup Fresh Tomato Bisque 2 lb. ripe tomatoes (about 6) 1 o